# Cohort Quality Deep Dive

This notebook drills deeper into the slide claim: **"deep discounting destroys cohort quality."**

The goal is to move beyond the annual headline and test alternative angles:

- Is the revenue-per-customer decline still visible after excluding shipping?
- Is the issue discounting in general, or **discount depth**?
- Are low-quality cohorts coming from channel/store/product mix shifts?
- Do discounted customers repeat, and do they repeat at full price?
- Is the annual average distorted by whales/outliers?

This notebook uses the corrected medallion datasets:

- Silver order revenue excludes shipping through `order_revenue_sgd`.
- `orders.line_items` and `orders.line_item_skus` are arrays, not joined strings.
- Gold customer revenue reconciles to Silver non-shipping order revenue.


In [1]:
from pathlib import Path
import warnings

# Keep Matplotlib/font caches inside the writable project area for sandboxed runs.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data" / "silver").exists():
    for parent in Path.cwd().parents:
        if (parent / "data" / "silver").exists():
            PROJECT_ROOT = parent
            break

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.options.display.float_format = "{:,.3f}".format

SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"
OUTPUT_DIR = GOLD_DIR / "cohort_quality_deepdive"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

orders = pd.read_parquet(SILVER_DIR / "orders.parquet")
lines = pd.read_parquet(SILVER_DIR / "lines.parquet")
customers = pd.read_parquet(GOLD_DIR / "customers.parquet")
order_lines = pd.read_parquet(GOLD_DIR / "order_lines_enriched.parquet")

orders["order_date"] = pd.to_datetime(orders["order_date"])
customers["first_order_date"] = pd.to_datetime(customers["first_order_date"])
customers["last_order_date"] = pd.to_datetime(customers["last_order_date"])

analysis_max_date = orders["order_date"].max()
print("Project root:", PROJECT_ROOT)
print("Analysis max order date:", analysis_max_date)
print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Output folder:", OUTPUT_DIR)

assert np.isclose(
    orders["order_revenue_sgd"],
    orders["order_total_incl_shipping_sgd"] - orders["shipping_revenue_sgd"],
    rtol=0,
    atol=0.01,
).all(), "Order revenue must exclude shipping."
assert np.isclose(customers["total_revenue_sgd"].sum(), orders["order_revenue_sgd"].sum(), rtol=0, atol=0.01), "Gold customer revenue must reconcile to Silver orders."

try:
    from IPython.display import display, HTML
except Exception:
    class HTML(str):
        pass
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string(index=False))
        else:
            print(obj)


Project root: /Users/tohzhengfeng/Documents/SMU/Academia/MITB 2526 Apr/ISSS603 Customer Analytics/Project
Analysis max order date: 2026-03-31 00:00:00+08:00
Orders: (27350, 49)
Customers: (13780, 17)
Output folder: /Users/tohzhengfeng/Documents/SMU/Academia/MITB 2526 Apr/ISSS603 Customer Analytics/Project/data/gold/cohort_quality_deepdive


## 1. Build Cohort-Level Customer Features

The slide uses calendar-year performance. For cohort quality, the safer view is **same-age customer behavior**: compare customers at the same number of days after first purchase.

Important definitions:

- `first_disc_depth = first_order_discount / (first_order_revenue + first_order_discount)`.
- `y1_revenue_sgd` is customer revenue in the first 365 days after acquisition, excluding shipping.
- `repeat_365d` means the customer placed another order within 365 days after first order.
- `full_price_repeat_365d` asks whether the customer came back without needing another discount.


In [2]:
orders = orders.sort_values(["customer_id", "order_date", "order_id"]).copy()
orders["order_year"] = orders["order_date"].dt.year
orders["order_gross_ex_shipping_sgd"] = orders["order_revenue_sgd"].fillna(0) + orders["order_discount_sgd"].fillna(0)
orders["discount_depth"] = np.where(
    orders["order_gross_ex_shipping_sgd"] > 0,
    orders["order_discount_sgd"].fillna(0) / orders["order_gross_ex_shipping_sgd"],
    0,
).clip(0, 1)
orders["is_discounted_order"] = orders["order_discount_sgd"].fillna(0) > 0

first_orders = (
    orders.groupby("customer_id", as_index=False)
    .head(1)
    [[
        "customer_id", "order_id", "order_date", "order_revenue_sgd", "order_discount_sgd",
        "order_gross_ex_shipping_sgd", "discount_depth", "channel", "store", "product_category",
        "line_item_count", "line_items", "line_item_skus",
    ]]
    .rename(columns={
        "order_id": "first_order_id",
        "order_date": "first_order_date",
        "order_revenue_sgd": "first_order_revenue_sgd",
        "order_discount_sgd": "first_order_discount_sgd",
        "order_gross_ex_shipping_sgd": "first_order_gross_ex_shipping_sgd",
        "discount_depth": "first_discount_depth",
        "channel": "first_channel",
        "store": "first_store",
        "product_category": "first_product_category",
        "line_item_count": "first_line_item_count",
        "line_items": "first_line_items",
        "line_item_skus": "first_line_item_skus",
    })
)
first_orders["cohort_year"] = first_orders["first_order_date"].dt.year
first_orders["first_discounted"] = first_orders["first_order_discount_sgd"].fillna(0) > 0

bins = [-0.001, 0, 0.10, 0.20, 0.30, 1.01]
labels = ["Full price", "1-10%", "11-20%", "21-30%", "31%+"]
first_orders["first_discount_band"] = pd.cut(first_orders["first_discount_depth"], bins=bins, labels=labels)

# Vectorized lifecycle windows: join every order to its customer's first date.
order_lifecycle = orders.merge(
    first_orders[["customer_id", "first_order_date"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)
order_lifecycle["days_since_first"] = (order_lifecycle["order_date"] - order_lifecycle["first_order_date"]).dt.days
order_lifecycle["is_after_first"] = order_lifecycle["days_since_first"] > 0
order_lifecycle["in_365d"] = order_lifecycle["days_since_first"].between(0, 365)
order_lifecycle["in_180d"] = order_lifecycle["days_since_first"].between(0, 180)
order_lifecycle["repeat_order_365d"] = order_lifecycle["is_after_first"] & (order_lifecycle["days_since_first"] <= 365)
order_lifecycle["repeat_order_180d"] = order_lifecycle["is_after_first"] & (order_lifecycle["days_since_first"] <= 180)
order_lifecycle["full_price_repeat_order_365d"] = order_lifecycle["repeat_order_365d"] & (order_lifecycle["order_discount_sgd"].fillna(0) <= 0)
order_lifecycle["full_price_repeat_order_180d"] = order_lifecycle["repeat_order_180d"] & (order_lifecycle["order_discount_sgd"].fillna(0) <= 0)

window_365 = (
    order_lifecycle[order_lifecycle["in_365d"]]
    .groupby("customer_id")
    .agg(
        y1_revenue_sgd=("order_revenue_sgd", "sum"),
        y1_orders=("order_id", "count"),
        y1_discounted_order_share=("is_discounted_order", "mean"),
    )
    .reset_index()
)
window_180 = (
    order_lifecycle[order_lifecycle["in_180d"]]
    .groupby("customer_id")
    .agg(
        y180_revenue_sgd=("order_revenue_sgd", "sum"),
        y180_orders=("order_id", "count"),
    )
    .reset_index()
)
repeat_flags = (
    order_lifecycle.groupby("customer_id")
    .agg(
        repeat_365d=("repeat_order_365d", "any"),
        repeat_180d=("repeat_order_180d", "any"),
        full_price_repeat_365d=("full_price_repeat_order_365d", "any"),
        full_price_repeat_180d=("full_price_repeat_order_180d", "any"),
    )
    .reset_index()
)

cohort_customers = (
    first_orders
    .merge(window_365, on="customer_id", how="left")
    .merge(window_180, on="customer_id", how="left")
    .merge(repeat_flags, on="customer_id", how="left")
)
cohort_customers["eligible_365d"] = cohort_customers["first_order_date"] <= analysis_max_date - pd.Timedelta(days=365)
cohort_customers["eligible_180d"] = cohort_customers["first_order_date"] <= analysis_max_date - pd.Timedelta(days=180)
for col in ["y1_revenue_sgd", "y1_orders", "y1_discounted_order_share", "y180_revenue_sgd", "y180_orders"]:
    cohort_customers[col] = cohort_customers[col].fillna(0)
for col in ["repeat_365d", "repeat_180d", "full_price_repeat_365d", "full_price_repeat_180d"]:
    cohort_customers[col] = cohort_customers[col].fillna(False)

cohort_customers.to_parquet(OUTPUT_DIR / "cohort_customer_features.parquet", index=False)
print("Cohort customer features:", cohort_customers.shape)
cohort_customers.head()


Cohort customer features: (13780, 27)


,customer_id,first_order_id,first_order_date,first_order_revenue_sgd,first_order_discount_sgd,first_order_gross_ex_shipping_sgd,first_discount_depth,first_channel,first_store,first_product_category,first_line_item_count,first_line_items,first_line_item_skus,cohort_year,first_discounted,first_discount_band,y1_revenue_sgd,y1_orders,y1_discounted_order_share,y180_revenue_sgd,y180_orders,repeat_365d,repeat_180d,full_price_repeat_365d,full_price_repeat_180d,eligible_365d,eligible_180d
0,6327387259135,4992506265855,2022-06-29 00:00:00+08:00,"1,693.120",0.000,"1,693.120",0.000,Subscription,SG,Unknown,1,"[16 x Plant Protein - 480g Pack, Natural (Unfl...",[],2022,False,Full price,"1,693.120",1,0.000,"1,693.120",1,False,False,False,False,True,True
1,6327388733695,4992632127743,2021-10-04 00:00:00+08:00,222.200,0.000,222.200,0.000,Subscription,SG,Unknown,1,"[2 x Prime Whey Isolate - 1KG Pack, Chocolate]",[],2021,False,Full price,222.200,1,0.000,222.200,1,False,False,False,False,True,True
2,6327388799231,4992730300671,2021-02-05 00:00:00+08:00,19.000,0.000,19.000,0.000,Subscription,SG,Unknown,1,"[1 x Plant Protein - 480g Pack, Uji Matcha]",[],2021,False,Full price,88.170,3,0.000,19.000,1,True,False,True,False,True,True
3,6327389520127,4992722010367,2021-03-04 00:00:00+08:00,29.000,0.000,29.000,0.000,Subscription,SG,Unknown,3,"[1 x Starter Pack, 1 x LushProtein Clear Shake...",[],2021,False,Full price,29.000,1,0.000,29.000,1,False,False,False,False,True,True
4,6327390699775,4992793739519,2020-08-18 00:00:00+08:00,139.900,0.000,139.900,0.000,Subscription,SG,Unknown,4,"[1 x Better Whey - 1KG Pack, Teh Tarik, 1 x Lu...",[],2020,False,Full price,492.980,2,0.000,492.980,2,True,True,True,True,True,True


## 2. Reframe The Annual Headline

Before drilling into causes, first rebuild the annual view with the corrected non-shipping revenue. Then check the distribution, not only the average.

Why this matters: if early years had a few large wholesale-like customers, the mean revenue per customer can overstate cohort quality. If median and upper-quartile values also fall, then the quality issue is broader than a whale/outlier story.


In [3]:
annual = (
    orders.groupby("order_year")
    .agg(
        orders=("order_id", "count"),
        customers=("customer_id", "nunique"),
        revenue_sgd=("order_revenue_sgd", "sum"),
        discounts_sgd=("order_discount_sgd", "sum"),
        shipping_sgd=("shipping_revenue_sgd", "sum"),
    )
    .reset_index()
    .rename(columns={"order_year": "year"})
)
annual["gross_ex_shipping_sgd"] = annual["revenue_sgd"] + annual["discounts_sgd"]
annual["discount_rate"] = annual["discounts_sgd"] / annual["gross_ex_shipping_sgd"].replace(0, np.nan)
annual["revenue_per_customer_sgd"] = annual["revenue_sgd"] / annual["customers"]
annual["orders_per_customer"] = annual["orders"] / annual["customers"]
annual.to_csv(OUTPUT_DIR / "annual_discount_quality_headline.csv", index=False)

customer_year = (
    orders.groupby(["order_year", "customer_id"])
    .agg(
        customer_year_revenue_sgd=("order_revenue_sgd", "sum"),
        customer_year_orders=("order_id", "count"),
        customer_year_discount_sgd=("order_discount_sgd", "sum"),
    )
    .reset_index()
    .rename(columns={"order_year": "year"})
)

distribution = (
    customer_year.groupby("year")
    .agg(
        mean_revenue_sgd=("customer_year_revenue_sgd", "mean"),
        median_revenue_sgd=("customer_year_revenue_sgd", "median"),
        p75_revenue_sgd=("customer_year_revenue_sgd", lambda s: s.quantile(0.75)),
        p90_revenue_sgd=("customer_year_revenue_sgd", lambda s: s.quantile(0.90)),
        p95_revenue_sgd=("customer_year_revenue_sgd", lambda s: s.quantile(0.95)),
    )
    .reset_index()
)

concentration_rows = []
for year, group in customer_year.groupby("year"):
    revenue = group["customer_year_revenue_sgd"].sort_values(ascending=False)
    n_top10 = max(int(np.ceil(len(revenue) * 0.10)), 1)
    n_top1 = max(int(np.ceil(len(revenue) * 0.01)), 1)
    concentration_rows.append({
        "year": year,
        "top_10pct_revenue_share": revenue.head(n_top10).sum() / revenue.sum(),
        "top_1pct_revenue_share": revenue.head(n_top1).sum() / revenue.sum(),
    })
concentration = pd.DataFrame(concentration_rows)
annual_distribution = annual.merge(distribution, on="year", how="left").merge(concentration, on="year", how="left")
annual_distribution.to_csv(OUTPUT_DIR / "annual_customer_value_distribution.csv", index=False)

annual_distribution


,year,orders,customers,revenue_sgd,discounts_sgd,shipping_sgd,gross_ex_shipping_sgd,discount_rate,revenue_per_customer_sgd,orders_per_customer,mean_revenue_sgd,median_revenue_sgd,p75_revenue_sgd,p90_revenue_sgd,p95_revenue_sgd,top_10pct_revenue_share,top_1pct_revenue_share
0,2020,2845,1695,"449,377.731",0.000,"7,126.712","449,377.731",0.000,265.120,1.678,265.120,81.758,188.280,520.673,975.590,0.625,0.237
1,2021,6243,3802,"837,100.606",0.000,"9,215.164","837,100.606",0.000,220.174,1.642,220.174,59.394,165.286,511.683,906.775,0.626,0.219
2,2022,3728,2308,"742,816.285","30,061.451","6,740.718","772,877.736",0.039,321.844,1.615,321.844,93.103,245.093,672.889,"1,246.201",0.624,0.238
3,2023,2250,1399,"178,525.472","32,323.921","2,976.583","210,849.392",0.153,127.609,1.608,127.609,78.255,139.430,277.918,407.376,0.390,0.086
4,2024,4260,2622,"280,414.194","127,496.038","4,529.301","407,910.232",0.313,106.947,1.625,106.947,68.727,121.455,233.451,327.133,0.394,0.100
5,2025,6400,4104,"403,564.227","221,772.297","5,014.750","625,336.524",0.355,98.334,1.559,98.334,53.180,110.528,207.000,292.379,0.496,0.210
6,2026,1624,1239,"183,940.810","52,530.520","1,609.230","236,471.330",0.222,148.459,1.311,148.459,88.760,141.100,226.030,314.941,0.469,0.272


## 3. Same-Age Cohort Quality

Annual totals mix acquisition age. A 2025 customer naturally has less time to repurchase than a 2021 customer. This section compares cohorts at the same age, mainly 365 days where enough follow-up exists.


In [4]:
cohort_quality = (
    cohort_customers.groupby("cohort_year")
    .agg(
        acquired_customers=("customer_id", "count"),
        pct_first_order_discounted=("first_discounted", "mean"),
        avg_first_discount_depth=("first_discount_depth", "mean"),
        avg_first_order_revenue_sgd=("first_order_revenue_sgd", "mean"),
        median_first_order_revenue_sgd=("first_order_revenue_sgd", "median"),
        eligible_365d=("eligible_365d", "sum"),
        repeat_365d_all=("repeat_365d", "mean"),
        avg_y1_revenue_all=("y1_revenue_sgd", "mean"),
    )
    .reset_index()
)

eligible365 = cohort_customers[cohort_customers["eligible_365d"]].copy()
y1_quality = (
    eligible365.groupby("cohort_year")
    .agg(
        eligible_customers=("customer_id", "count"),
        repeat_365d=("repeat_365d", "mean"),
        full_price_repeat_365d=("full_price_repeat_365d", "mean"),
        avg_y1_revenue_sgd=("y1_revenue_sgd", "mean"),
        median_y1_revenue_sgd=("y1_revenue_sgd", "median"),
        avg_first_discount_depth=("first_discount_depth", "mean"),
    )
    .reset_index()
)
cohort_quality = cohort_quality.merge(y1_quality, on="cohort_year", how="left", suffixes=("", "_eligible"))
cohort_quality.to_csv(OUTPUT_DIR / "same_age_cohort_quality.csv", index=False)
cohort_quality


,cohort_year,acquired_customers,pct_first_order_discounted,avg_first_discount_depth,avg_first_order_revenue_sgd,median_first_order_revenue_sgd,eligible_365d,repeat_365d_all,avg_y1_revenue_all,eligible_customers,repeat_365d,full_price_repeat_365d,avg_y1_revenue_sgd,median_y1_revenue_sgd,avg_first_discount_depth_eligible
0,2020,1695,0.000,0.000,121.128,57.273,1695,0.379,328.963,"1,695.000",0.379,0.379,328.963,96.667,0.000
1,2021,3321,0.000,0.000,95.239,35.636,3321,0.281,209.311,"3,321.000",0.281,0.279,209.311,56.061,0.000
2,2022,1477,0.137,0.038,135.239,51.355,1477,0.261,238.211,"1,477.000",0.261,0.191,238.211,73.990,0.038
3,2023,874,0.610,0.112,64.682,53.915,874,0.257,104.789,874.000,0.257,0.175,104.789,67.301,0.112
4,2024,2015,0.815,0.287,58.187,53.330,2015,0.281,97.336,"2,015.000",0.281,0.129,97.336,62.110,0.287
5,2025,3519,0.616,0.287,57.099,35.900,260,0.203,100.790,260.000,0.238,0.142,291.298,65.985,0.248
6,2026,879,0.694,0.157,97.374,70.570,0,0.084,108.184,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Discount Depth: The More Precise Cut

The key drill-down is not simply discounted vs non-discounted. The sharper question is: **at what discount depth does cohort quality break?**

This table uses only customers with a full 365-day observation window, so cohorts are compared at the same lifecycle age.


In [5]:
discount_depth_y1 = (
    eligible365.groupby(["cohort_year", "first_discount_band"], observed=True)
    .agg(
        customers=("customer_id", "count"),
        repeat_365d=("repeat_365d", "mean"),
        full_price_repeat_365d=("full_price_repeat_365d", "mean"),
        avg_y1_revenue_sgd=("y1_revenue_sgd", "mean"),
        median_y1_revenue_sgd=("y1_revenue_sgd", "median"),
        avg_first_order_revenue_sgd=("first_order_revenue_sgd", "mean"),
    )
    .reset_index()
)
discount_depth_y1.to_csv(OUTPUT_DIR / "discount_depth_y1_quality.csv", index=False)
discount_depth_y1


,cohort_year,first_discount_band,customers,repeat_365d,full_price_repeat_365d,avg_y1_revenue_sgd,median_y1_revenue_sgd,avg_first_order_revenue_sgd
0,2020,Full price,1695,0.379,0.379,328.963,96.667,121.128
1,2021,Full price,3321,0.281,0.279,209.311,56.061,95.239
2,2022,Full price,1275,0.267,0.215,256.183,70.127,143.520
3,2022,1-10%,67,0.269,0.030,145.879,92.064,96.268
4,2022,11-20%,28,0.179,0.036,108.886,85.041,74.586
5,2022,21-30%,40,0.200,0.050,139.726,82.934,85.429
6,2022,31%+,67,0.209,0.045,101.380,73.912,71.705
7,2023,Full price,341,0.355,0.311,103.872,59.900,52.489
8,2023,1-10%,193,0.187,0.083,104.491,77.670,76.444
9,2023,11-20%,182,0.187,0.088,98.538,68.230,69.106


## 5. Channel, Store, And Product Mix

If discounting increased at the same time the business changed channels, products, or markets, the headline may hide mix effects. These cuts help identify whether lower quality comes from **who was acquired**, not only from the offer used to acquire them.


In [6]:
channel_quality = (
    eligible365.groupby(["cohort_year", "first_channel"])
    .agg(
        customers=("customer_id", "count"),
        mix_share=("customer_id", "count"),
        repeat_365d=("repeat_365d", "mean"),
        full_price_repeat_365d=("full_price_repeat_365d", "mean"),
        avg_y1_revenue_sgd=("y1_revenue_sgd", "mean"),
        avg_first_discount_depth=("first_discount_depth", "mean"),
    )
    .reset_index()
)
channel_quality["mix_share"] = channel_quality["customers"] / channel_quality.groupby("cohort_year")["customers"].transform("sum")
channel_quality.to_csv(OUTPUT_DIR / "channel_mix_quality.csv", index=False)

store_quality = (
    cohort_customers.groupby(["cohort_year", "first_store"])
    .agg(
        customers=("customer_id", "count"),
        mix_share=("customer_id", "count"),
        avg_first_discount_depth=("first_discount_depth", "mean"),
        avg_first_order_revenue_sgd=("first_order_revenue_sgd", "mean"),
    )
    .reset_index()
)
store_quality["mix_share"] = store_quality["customers"] / store_quality.groupby("cohort_year")["customers"].transform("sum")
store_quality.to_csv(OUTPUT_DIR / "store_mix_quality.csv", index=False)

product_quality = (
    eligible365.groupby(["cohort_year", "first_product_category"])
    .agg(
        customers=("customer_id", "count"),
        mix_share=("customer_id", "count"),
        repeat_365d=("repeat_365d", "mean"),
        full_price_repeat_365d=("full_price_repeat_365d", "mean"),
        avg_y1_revenue_sgd=("y1_revenue_sgd", "mean"),
        avg_first_discount_depth=("first_discount_depth", "mean"),
    )
    .reset_index()
)
product_quality["mix_share"] = product_quality["customers"] / product_quality.groupby("cohort_year")["customers"].transform("sum")
product_quality.to_csv(OUTPUT_DIR / "product_mix_quality.csv", index=False)

print("Channel quality, recent eligible cohorts:")
display(channel_quality[channel_quality["cohort_year"].isin([2023, 2024, 2025])].sort_values(["cohort_year", "customers"], ascending=[True, False]))
print("Store quality:")
display(store_quality)
print("Product quality, recent eligible cohorts:")
display(product_quality[product_quality["cohort_year"].isin([2023, 2024, 2025])].sort_values(["cohort_year", "customers"], ascending=[True, False]))


Channel quality, recent eligible cohorts:


,cohort_year,first_channel,customers,mix_share,repeat_365d,full_price_repeat_365d,avg_y1_revenue_sgd,avg_first_discount_depth
9,2023,Subscription,324,0.371,0.444,0.318,149.490,0.122
7,2023,Direct / Organic,293,0.335,0.198,0.092,85.244,0.136
8,2023,Marketplace,257,0.294,0.089,0.089,70.718,0.074
13,2024,Subscription,973,0.483,0.353,0.201,119.062,0.306
10,2024,Direct / Organic,754,0.374,0.255,0.080,85.090,0.318
12,2024,Marketplace,287,0.142,0.111,0.010,55.995,0.141
11,2024,Email,1,0.000,0.000,0.000,57.273,0.000
15,2025,Direct / Organic,137,0.527,0.241,0.146,137.780,0.301
17,2025,Marketplace,79,0.304,0.127,0.013,618.888,0.258
19,2025,Subscription,33,0.127,0.394,0.333,137.033,0.074


Store quality:


,cohort_year,first_store,customers,mix_share,avg_first_discount_depth,avg_first_order_revenue_sgd
0,2020,MY,897,0.529,0.000,75.393
1,2020,SG,798,0.471,0.000,172.537
2,2021,MY,2049,0.617,0.000,92.406
3,2021,SG,1272,0.383,0.000,99.803
4,2022,MY,854,0.578,0.039,113.031
5,2022,SG,623,0.422,0.037,165.681
6,2023,MY,368,0.421,0.117,56.258
7,2023,SG,506,0.579,0.109,70.809
8,2024,MY,952,0.472,0.287,54.699
9,2024,SG,1063,0.528,0.287,61.310


Product quality, recent eligible cohorts:


,cohort_year,first_product_category,customers,mix_share,repeat_365d,full_price_repeat_365d,avg_y1_revenue_sgd,avg_first_discount_depth
11,2023,Other,563,0.644,0.238,0.167,108.605,0.103
14,2023,Unknown,138,0.158,0.268,0.196,105.526,0.089
10,2023,Collagen Glow,98,0.112,0.306,0.194,87.466,0.162
12,2023,Soy Protein,44,0.050,0.364,0.182,103.951,0.160
8,2023,Accessories,27,0.031,0.222,0.148,81.338,0.168
13,2023,Supplements,3,0.003,0.333,0.333,107.800,0.080
9,2023,Clear Protein,1,0.001,1.000,0.000,213.810,0.050
19,2024,Other,696,0.345,0.307,0.124,92.955,0.252
16,2024,Clear Protein,638,0.317,0.255,0.124,107.675,0.269
18,2024,Lean Protein,192,0.095,0.318,0.151,107.310,0.375


## 6. Mix vs Quality Decomposition

This is a useful slide angle: separate whether quality changed because the customer mix changed, or because customers inside each segment got worse.

The function below decomposes the change in average Year-1 revenue between two cohorts:

- **Mix effect:** what would have happened if the later cohort had the earlier cohort's segment quality but the later cohort's mix?
- **Within-segment quality effect:** what changed because each segment itself became better/worse?

This is directional, not causal, but it is a stronger diagnostic than a single annual chart.


In [7]:
def decompose_mix_quality(df, segment_col, metric_col, base_year, compare_year):
    base = df[df["cohort_year"] == base_year]
    comp = df[df["cohort_year"] == compare_year]
    segs = sorted(set(base[segment_col].dropna().astype(str)) | set(comp[segment_col].dropna().astype(str)))

    def stats(group):
        out = {}
        total = len(group)
        for seg in segs:
            g = group[group[segment_col].astype(str) == seg]
            out[seg] = {
                "share": len(g) / total if total else 0,
                "quality": g[metric_col].mean() if len(g) else 0,
            }
        return out

    b = stats(base)
    c = stats(comp)
    base_avg = base[metric_col].mean()
    comp_avg = comp[metric_col].mean()
    mix_effect = sum((c[s]["share"] - b[s]["share"]) * b[s]["quality"] for s in segs)
    quality_effect = sum(c[s]["share"] * (c[s]["quality"] - b[s]["quality"]) for s in segs)

    detail = pd.DataFrame([
        {
            "segment": s,
            "base_share": b[s]["share"],
            "compare_share": c[s]["share"],
            "base_quality": b[s]["quality"],
            "compare_quality": c[s]["quality"],
        }
        for s in segs
    ])
    summary = pd.DataFrame([{
        "segment_col": segment_col,
        "metric": metric_col,
        "base_year": base_year,
        "compare_year": compare_year,
        "base_avg": base_avg,
        "compare_avg": comp_avg,
        "actual_change": comp_avg - base_avg,
        "mix_effect": mix_effect,
        "within_segment_quality_effect": quality_effect,
        "residual_check": (comp_avg - base_avg) - mix_effect - quality_effect,
    }])
    return summary, detail

# Use 2022 -> 2024 because both have full 365-day follow-up and include the discounting era.
decomp_channel, decomp_channel_detail = decompose_mix_quality(eligible365, "first_channel", "y1_revenue_sgd", 2022, 2024)
decomp_discount, decomp_discount_detail = decompose_mix_quality(eligible365, "first_discount_band", "y1_revenue_sgd", 2022, 2024)

decomp_summary = pd.concat([decomp_channel, decomp_discount], ignore_index=True)
decomp_summary.to_csv(OUTPUT_DIR / "mix_quality_decomposition_summary.csv", index=False)
decomp_channel_detail.to_csv(OUTPUT_DIR / "mix_quality_decomposition_channel_detail.csv", index=False)
decomp_discount_detail.to_csv(OUTPUT_DIR / "mix_quality_decomposition_discount_detail.csv", index=False)

display(decomp_summary)
print("Channel decomposition detail:")
display(decomp_channel_detail)
print("Discount-depth decomposition detail:")
display(decomp_discount_detail)


,segment_col,metric,base_year,compare_year,base_avg,compare_avg,actual_change,mix_effect,within_segment_quality_effect,residual_check
0,first_channel,y1_revenue_sgd,2022,2024,238.211,97.336,-140.875,-5.055,-135.820,-0.000
1,first_discount_band,y1_revenue_sgd,2022,2024,238.211,97.336,-140.875,-93.786,-47.089,0.000


Channel decomposition detail:


,segment,base_share,compare_share,base_quality,compare_quality
0,Direct / Organic,0.578,0.374,210.032,85.090
1,Email,0.000,0.000,0.000,57.273
2,Marketplace,0.023,0.142,114.946,55.995
3,Subscription,0.399,0.483,286.183,119.062


Discount-depth decomposition detail:


,segment,base_share,compare_share,base_quality,compare_quality
0,1-10%,0.045,0.137,145.879,134.246
1,11-20%,0.019,0.177,108.886,108.405
2,21-30%,0.027,0.181,139.726,106.212
3,31%+,0.045,0.320,101.380,61.083
4,Full price,0.863,0.185,256.183,113.269


## 7. Story Charts

These charts turn the tables above into a presentation-ready story flow. They are generated as SVGs so the notebook can render reliably without Matplotlib/font-cache dependencies.


In [8]:
from html import escape

CHART_DIR = OUTPUT_DIR / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

C = {
    "teal": "#13b889", "blue": "#1676bf", "orange": "#c17700",
    "red": "#e23b2e", "dark": "#222222", "gray": "#666666", "grid": "#d9d9d9"
}

def sc(v, a, b, x, y):
    return (x + y) / 2 if a == b else x + (v - a) * (y - x) / (b - a)

def save_svg(name, svg):
    path = CHART_DIR / name
    path.write_text(svg, encoding="utf-8")
    display(HTML(svg))
    print("saved:", path.relative_to(PROJECT_ROOT))
    return path

def headline_svg(df):
    d = df[(df.year >= 2020) & (df.year <= 2025)].copy()
    w,h,ml,mt,pw,ph = 980,500,90,75,800,330
    years = d.year.astype(int).tolist()
    xs = [sc(i,0,len(years)-1,ml+40,ml+pw-40) for i in range(len(years))]
    max_c = d.customers.max()*1.12
    max_rpc = d.revenue_per_customer_sgd.max()*1.15
    max_disc = max(d.discount_rate.max()*1.25, .4)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" viewBox="0 0 {w} {h}"><rect width="100%" height="100%" fill="white"/>']
    parts += [f'<text x="{ml}" y="35" font-size="26" font-weight="700" fill="{C["dark"]}">More customers, lower customer quality</text>',
              f'<text x="{ml}" y="58" font-size="15" fill="{C["gray"]}">Bars = customers; orange = revenue/customer; red = discount rate. Revenue excludes shipping.</text>']
    for i in range(5):
        y=mt+ph*i/4; parts.append(f'<line x1="{ml}" y1="{y}" x2="{ml+pw}" y2="{y}" stroke="{C["grid"]}"/>')
    rpc_pts=[]; disc_pts=[]
    for x,(_,r) in zip(xs,d.iterrows()):
        bh=sc(r.customers,0,max_c,0,ph)
        parts.append(f'<rect x="{x-20}" y="{mt+ph-bh}" width="40" height="{bh}" fill="{C["blue"]}" opacity="0.85"/>')
        parts.append(f'<text x="{x}" y="{mt+ph+28}" text-anchor="middle" font-size="15" fill="{C["gray"]}">{int(r.year)}</text>')
        rpc_pts.append((x, sc(r.revenue_per_customer_sgd,0,max_rpc,mt+ph,mt)))
        disc_pts.append((x, sc(r.discount_rate,0,max_disc,mt+ph,mt)))
    parts.append('<polyline points="' + ' '.join(f'{x:.1f},{y:.1f}' for x,y in rpc_pts) + f'" fill="none" stroke="{C["orange"]}" stroke-width="4"/>')
    parts.append('<polyline points="' + ' '.join(f'{x:.1f},{y:.1f}' for x,y in disc_pts) + f'" fill="none" stroke="{C["red"]}" stroke-width="4"/>')
    for x,y in rpc_pts: parts.append(f'<circle cx="{x}" cy="{y}" r="5" fill="{C["orange"]}"/>')
    for x,y in disc_pts: parts.append(f'<circle cx="{x}" cy="{y}" r="5" fill="{C["red"]}"/>')
    r22=d.loc[d.year==2022].iloc[0]; r25=d.loc[d.year==2025].iloc[0]
    parts.append(f'<text x="{ml}" y="{h-22}" font-size="14" fill="{C["dark"]}">2022 to 2025: revenue/customer fell from SGD {r22.revenue_per_customer_sgd:.0f} to SGD {r25.revenue_per_customer_sgd:.0f}; discount rate rose from {r22.discount_rate:.1%} to {r25.discount_rate:.1%}.</text></svg>')
    return ''.join(parts)

def y1_svg(df):
    d=df[df.cohort_year.between(2020,2025)].copy()
    w,h,ml,mt,pw,ph=900,430,80,70,760,280
    xs=[sc(i,0,len(d)-1,ml+40,ml+pw-40) for i in range(len(d))]
    max_rev=d.avg_y1_revenue_sgd.max()*1.15; max_rep=max(d.full_price_repeat_365d.max()*1.3,.4)
    parts=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" viewBox="0 0 {w} {h}"><rect width="100%" height="100%" fill="white"/>',
           f'<text x="{ml}" y="35" font-size="24" font-weight="700" fill="{C["dark"]}">Same-age cohort quality has deteriorated</text>',
           f'<text x="{ml}" y="57" font-size="14" fill="{C["gray"]}">Bars = average Year-1 revenue; line = full-price repeat within 365 days.</text>']
    pts=[]
    for x,(_,r) in zip(xs,d.iterrows()):
        bh=sc(r.avg_y1_revenue_sgd,0,max_rev,0,ph)
        parts.append(f'<rect x="{x-24}" y="{mt+ph-bh}" width="48" height="{bh}" fill="{C["teal"]}"/>')
        parts.append(f'<text x="{x}" y="{mt+ph+28}" text-anchor="middle" font-size="15" fill="{C["gray"]}">{int(r.cohort_year)}</text>')
        pts.append((x,sc(r.full_price_repeat_365d,0,max_rep,mt+ph,mt)))
    parts.append('<polyline points="'+' '.join(f'{x:.1f},{y:.1f}' for x,y in pts)+f'" fill="none" stroke="{C["orange"]}" stroke-width="4"/>')
    for x,y in pts: parts.append(f'<circle cx="{x}" cy="{y}" r="5" fill="{C["orange"]}"/>')
    r22=d.loc[d.cohort_year==2022].iloc[0]; r24=d.loc[d.cohort_year==2024].iloc[0]
    parts.append(f'<text x="{ml}" y="{h-20}" font-size="14" fill="{C["dark"]}">2022 to 2024: average Year-1 revenue fell from SGD {r22.avg_y1_revenue_sgd:.0f} to SGD {r24.avg_y1_revenue_sgd:.0f}.</text></svg>')
    return ''.join(parts)

def discount_svg(df, year=2024):
    d=df[df.cohort_year==year].copy()
    order=["Full price","1-10%","11-20%","21-30%","31%+"]
    d.first_discount_band=pd.Categorical(d.first_discount_band, order, ordered=True); d=d.sort_values('first_discount_band')
    w,h,ml,mt,pw,ph=900,430,80,70,760,280
    xs=[sc(i,0,len(d)-1,ml+45,ml+pw-45) for i in range(len(d))]
    max_n=d.customers.max()*1.15; max_r=max(d.full_price_repeat_365d.max()*1.25,.35)
    parts=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" viewBox="0 0 {w} {h}"><rect width="100%" height="100%" fill="white"/>',
           f'<text x="{ml}" y="35" font-size="24" font-weight="700" fill="{C["dark"]}">Deep discounts fail the full-price repeat test</text>',
           f'<text x="{ml}" y="57" font-size="14" fill="{C["gray"]}">{year} cohort. Bars = customers; line = full-price repeat rate.</text>']
    pts=[]
    for x,(_,r) in zip(xs,d.iterrows()):
        bh=sc(r.customers,0,max_n,0,ph)
        parts.append(f'<rect x="{x-28}" y="{mt+ph-bh}" width="56" height="{bh}" fill="{C["blue"]}" opacity="0.82"/>')
        parts.append(f'<text x="{x}" y="{mt+ph+30}" text-anchor="middle" font-size="14" fill="{C["gray"]}">{escape(str(r.first_discount_band))}</text>')
        pts.append((x,sc(r.full_price_repeat_365d,0,max_r,mt+ph,mt)))
    parts.append('<polyline points="'+' '.join(f'{x:.1f},{y:.1f}' for x,y in pts)+f'" fill="none" stroke="{C["red"]}" stroke-width="4"/>')
    for x,y in pts: parts.append(f'<circle cx="{x}" cy="{y}" r="6" fill="{C["red"]}"/>')
    fp=d.loc[d.first_discount_band=='Full price'].iloc[0]; deep=d.loc[d.first_discount_band=='31%+'].iloc[0]
    parts.append(f'<text x="{ml}" y="{h-20}" font-size="14" fill="{C["dark"]}">Full price: {fp.full_price_repeat_365d:.1%} full-price repeat vs 31%+: {deep.full_price_repeat_365d:.1%}.</text></svg>')
    return ''.join(parts)

def decomp_svg(df):
    r=df[df.segment_col=='first_discount_band'].iloc[0]
    w,h,ml,mt,bar,gap=900,360,80,70,140,70
    ymin=min(0,r.mix_effect+r.within_segment_quality_effect)-20; ymax=max(r.base_avg,r.compare_avg)+40
    base_y=sc(0,ymin,ymax,h-70,mt); x=ml; cur=r.base_avg
    labels=['2022 Y1 Rev','Mix shift','Within-band quality','2024 Y1 Rev']; vals=[r.base_avg,r.mix_effect,r.within_segment_quality_effect,r.compare_avg]; cols=[C['teal'],C['red'],C['red'],C['blue']]
    parts=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" viewBox="0 0 {w} {h}"><rect width="100%" height="100%" fill="white"/>',
           f'<text x="{ml}" y="35" font-size="24" font-weight="700" fill="{C["dark"]}">What explains the Year-1 revenue collapse?</text>',
           f'<text x="{ml}" y="57" font-size="14" fill="{C["gray"]}">2022 to 2024 decomposition using first-order discount bands.</text>']
    for i,(lab,val,col) in enumerate(zip(labels,vals,cols)):
        if i==0 or i==3:
            y=sc(val,ymin,ymax,h-70,mt); bh=abs(base_y-y)
            parts.append(f'<rect x="{x}" y="{y}" width="{bar}" height="{bh}" fill="{col}"/>')
            parts.append(f'<text x="{x+bar/2}" y="{y-8}" text-anchor="middle" font-size="14" fill="{C["dark"]}">SGD {val:.0f}</text>')
        else:
            nxt=cur+val; y1=sc(cur,ymin,ymax,h-70,mt); y2=sc(nxt,ymin,ymax,h-70,mt); y=min(y1,y2); bh=abs(y2-y1)
            parts.append(f'<rect x="{x}" y="{y}" width="{bar}" height="{bh}" fill="{col}"/>')
            parts.append(f'<text x="{x+bar/2}" y="{y+bh+18}" text-anchor="middle" font-size="14" fill="{C["red"]}">SGD {val:.0f}</text>')
            cur=nxt
        parts.append(f'<text x="{x+bar/2}" y="{h-32}" text-anchor="middle" font-size="13" fill="{C["gray"]}">{lab}</text>')
        x += bar+gap
    parts.append(f'<text x="{ml}" y="{h-8}" font-size="14" fill="{C["dark"]}">Discount-depth mix explains SGD {abs(r.mix_effect):.0f}; within-band quality explains SGD {abs(r.within_segment_quality_effect):.0f}.</text></svg>')
    return ''.join(parts)


In [9]:
save_svg("01_headline_customers_revenue_discount.svg", headline_svg(annual_distribution))
save_svg("02_same_age_y1_cohort_quality.svg", y1_svg(cohort_quality))
save_svg("03_discount_depth_full_price_repeat.svg", discount_svg(discount_depth_y1, year=2024))
save_svg("04_discount_mix_decomposition.svg", decomp_svg(decomp_summary))


saved: data/gold/cohort_quality_deepdive/charts/01_headline_customers_revenue_discount.svg


saved: data/gold/cohort_quality_deepdive/charts/02_same_age_y1_cohort_quality.svg


saved: data/gold/cohort_quality_deepdive/charts/03_discount_depth_full_price_repeat.svg


saved: data/gold/cohort_quality_deepdive/charts/04_discount_mix_decomposition.svg


PosixPath('/Users/tohzhengfeng/Documents/SMU/Academia/MITB 2526 Apr/ISSS603 Customer Analytics/Project/data/gold/cohort_quality_deepdive/charts/04_discount_mix_decomposition.svg')

### Storyboard For The Slide Deck

- **Chart 1:** Start with the contradiction: more customers, lower value per customer, higher discount rate.
- **Chart 2:** Prove it is not only a calendar-age issue by comparing cohorts at Year 1.
- **Chart 3:** Show the mechanism: deep discount customers are much less likely to repeat at full price.
- **Chart 4:** Quantify the driver: the shift toward deeper discount bands explains most of the Year-1 revenue decline from 2022 to 2024.


## 8. Recommended Insight Angles For The Slide

Use these as candidate follow-up slides or speaker notes:

1. **Discount depth matters more than discount presence.** The strongest break appears in the deepest first-order discount bands, especially when measuring full-price repeat behavior.
2. **Revenue quality fell across the distribution, not only because of more customers.** Median and upper-quartile customer-year revenue give a useful check against average-only storytelling.
3. **Full-price repeat is the cleanest loyalty test.** A discounted customer who repeats only when discounted again is different from one who returns at full price.
4. **Same-age cohort analysis is stronger than calendar-year analysis.** Compare 2022 vs 2024 at Year 1, not 2025 vs 2021 at different lifecycle ages.
5. **Channel and store mix may be confounders.** Marketplace and Direct/Organic can carry very different repeat profiles; discount strategy and acquisition mix should be separated.
6. **Recharge/subscription customers deserve their own panel.** Subscription-origin cohorts often behave differently from one-time marketplace/direct buyers, so mixing them may hide the actual issue.

The output CSVs in `data/gold/cohort_quality_deepdive/` are designed to feed charts for these cuts.


In [10]:
print("Deep-dive outputs written to:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", path.relative_to(PROJECT_ROOT))
print(" -", (OUTPUT_DIR / "cohort_customer_features.parquet").relative_to(PROJECT_ROOT))


Deep-dive outputs written to: /Users/tohzhengfeng/Documents/SMU/Academia/MITB 2526 Apr/ISSS603 Customer Analytics/Project/data/gold/cohort_quality_deepdive
 - data/gold/cohort_quality_deepdive/annual_customer_value_distribution.csv
 - data/gold/cohort_quality_deepdive/annual_discount_quality_headline.csv
 - data/gold/cohort_quality_deepdive/channel_mix_quality.csv
 - data/gold/cohort_quality_deepdive/discount_depth_y1_quality.csv
 - data/gold/cohort_quality_deepdive/mix_quality_decomposition_channel_detail.csv
 - data/gold/cohort_quality_deepdive/mix_quality_decomposition_discount_detail.csv
 - data/gold/cohort_quality_deepdive/mix_quality_decomposition_summary.csv
 - data/gold/cohort_quality_deepdive/product_mix_quality.csv
 - data/gold/cohort_quality_deepdive/same_age_cohort_quality.csv
 - data/gold/cohort_quality_deepdive/store_mix_quality.csv
 - data/gold/cohort_quality_deepdive/cohort_customer_features.parquet
